In [51]:
import pandas as pd

df = pd.read_csv("../data/evaluation_mails_for_milestone2.csv")
df.head()


,email_text,expected_action,expected_tone
0,"Hi team, just a reminder that the Q3 strategy ...",notify,urgent
1,"Dear Student, your internship application for ...",notify,polite
2,Attached is the invoice for the web developmen...,respond,polite
3,"Hey, are we still on for lunch today? Let me k...",respond,neutral
4,Congratulations! You have been selected for th...,respond,polite


In [52]:
#clean the email text
import re
def clean_email(text):
    if isinstance(text, str):
        text = text.lower()
        text = re.sub(r"http\S+", "", text)
        text = re.sub(r"<.*?>", "", text)
        text = re.sub(r"[^a-zA-Z0-9 ]", " ", text)
        text = re.sub(r"\s+", " ", text).strip()
        return text
    else:
        return ""

df["clean_text"] = df["email_text"].apply(clean_email)
df.head()


,email_text,expected_action,expected_tone,clean_text
0,"Hi team, just a reminder that the Q3 strategy ...",notify,urgent,hi team just a reminder that the q3 strategy m...
1,"Dear Student, your internship application for ...",notify,polite,dear student your internship application for t...
2,Attached is the invoice for the web developmen...,respond,polite,attached is the invoice for the web developmen...
3,"Hey, are we still on for lunch today? Let me k...",respond,neutral,hey are we still on for lunch today let me kno...
4,Congratulations! You have been selected for th...,respond,polite,congratulations you have been selected for the...


In [53]:
# Email_assistant return dict 
def email_assistant(email_text):
    text = email_text.lower()

    urgent_keywords = ["urgent", "submit", "deadline"]
    polite_keywords = ["thanks", "thank you", "appreciate"]

    if any(word in text for word in urgent_keywords):
        return {
            "action": "notify",
            "tone": "urgent"
        }

    if any(word in text for word in polite_keywords):
        return {
            "action": "ignore",
            "tone": "polite"
        }

    return {
        "action": "respond",
        "tone": "neutral"
    }


    
df[["predicted_action", "predicted_tone"]] = (
    df["clean_text"]
    .apply(email_assistant)
    .apply(pd.Series)
)


In [54]:
df["correct_action"] = df["predicted_action"] == df["expected_action"]
df["correct_tone"] = df["predicted_tone"] == df["expected_tone"]

df.head()

,email_text,expected_action,expected_tone,clean_text,predicted_action,predicted_tone,correct_action,correct_tone
0,"Hi team, just a reminder that the Q3 strategy ...",notify,urgent,hi team just a reminder that the q3 strategy m...,respond,neutral,False,False
1,"Dear Student, your internship application for ...",notify,polite,dear student your internship application for t...,respond,neutral,False,False
2,Attached is the invoice for the web developmen...,respond,polite,attached is the invoice for the web developmen...,respond,neutral,True,False
3,"Hey, are we still on for lunch today? Let me k...",respond,neutral,hey are we still on for lunch today let me kno...,respond,neutral,True,True
4,Congratulations! You have been selected for th...,respond,polite,congratulations you have been selected for the...,respond,neutral,True,False


In [55]:
action_accuracy = df["correct_action"].mean() * 100
tone_accuracy = df["correct_tone"].mean() * 100

print("Action accuracy (%):", action_accuracy)
print("Tone accuracy (%):", tone_accuracy)

Action accuracy (%): 50.0
Tone accuracy (%): 54.0


In [56]:
wrong_df = df[~(df["correct_action"] & df["correct_tone"])]
print("\nRows with incorrect predictions: ", len(wrong_df))
wrong_df[
    ["clean_text", "expected_action", "predicted_action",
     "expected_tone", "predicted_tone"]
].head()


Rows with incorrect predictions:  81


,clean_text,expected_action,predicted_action,expected_tone,predicted_tone
0,hi team just a reminder that the q3 strategy m...,notify,respond,urgent,neutral
1,dear student your internship application for t...,notify,respond,polite,neutral
2,attached is the invoice for the web developmen...,respond,respond,polite,neutral
4,congratulations you have been selected for the...,respond,respond,polite,neutral
5,the server will be undergoing scheduled mainte...,notify,respond,neutral,neutral


In [57]:
path = "../data/milestone2_output_gajendra.csv"
wrong_df.to_csv(path, index=False)
print("File Saved")

File Saved


## **Q1. Which type of emails were hardest to classify?**

1. Emails with mixed intent or neutral language were the hardest to classify.

2. Emails that did not contain strong keywords (like “urgent” or “thanks”) were often ambiguous.

3. Some emails sounded polite but still required action, or mentioned deadlines indirectly.

4. Neutral, conversational emails without explicit cues were difficult for simple rules to interpret      correctly.

## **Q2. Why did your rules fail in some cases?**

The rules failed in some cases because they were primarily keyword-based and lacked contextual understanding. The system depended on the presence of specific words rather than the overall meaning of the email. As a result, it could not properly handle implied urgency, varied phrasing, or multiple intents within a single message. In some cases, the presence of a single keyword outweighed the broader context, leading to incorrect classifications. This rigidity limited the system’s ability to generalize to diverse or nuanced email styles.

## **Q3. How could an LLM improve this process ?**

A Large Language Model (LLM) could significantly improve this process by understanding context and semantics rather than relying on fixed rules. An LLM can interpret the meaning of entire sentences, recognize implied intent, and handle variations in language and tone more effectively. It can also manage emails with mixed signals by weighing context instead of matching keywords. Overall, an LLM would provide more accurate and human-like classifications, especially for ambiguous or previously unseen email patterns.

# Setting up LLM as a judge

In [58]:
from langsmith import Client

from dotenv import load_dotenv
load_dotenv()

client = Client()


#Define the judge prompt 

judge_prompt = """You are an evaluator. Compare the model output with the ideal answer.

Check:
1. Action Correctness
2. Tone Correctness

Give Score: 
1 = correct
2 = Not-correct

"""

In [59]:
# Run + judge

def evaluate(agent_output, ideal_action, ideal_tone):
    if (
        agent_output["action"] == ideal_action
        and agent_output["tone"] == ideal_tone
    ):
        return 1
    else:
        return 0
    


#Sample test

agent_output = {'action': 'notify', 'tone': 'urgent'}

ideal_action = 'notify'
ideal_tone = 'urgent'

score = evaluate(agent_output, ideal_action, ideal_tone)
print(score)


1


In [ ]:
# Read sample email file

df_2 = pd.read_csv("../data/sample_emails_with_triage_200.csv") 

# Cleaning email_text of sample file

df_2["clean_text"] = df_2["body"].apply(clean_email)
df_2.head()


# Adding predicted action and predicted_tone coloumn.
df_2[["predicted_action", "predicted_tone"]] = (
    df["clean_text"]
    .apply(email_assistant)
    .apply(pd.Series)
)



In [62]:
#Run the evaluation on sample data

scores = []
for _, row in df_2.iterrows():
    prediction = email_assistant(row['clean_text'])
    score = evaluate(
        prediction,
        row["ideal_intent"],
        row["ideal_tone"]
    )
    scores.append(score)

accuracy = (sum(scores) / len(scores)) * 100
accuracy



19.5

## Evaluating accuracy and merging score coloumn in the csv.

In [ ]:
def evaluate(row):
	score = 0
	if row["predicted_action"] == row["ideal_intent"]:
		score += 1
	if row["predicted_tone"] == row["ideal_tone"]:
		score += 1
	return score


# Using a copy of the existing dataframe that already contains predicted and expected cols.
eval_df = df.copy()
eval_df["score"] = eval_df.apply(evaluate, axis=1)
eval_df.head()
accuracy = (eval_df["score"].sum() / (len(eval_df) * 2)) * 100
print(accuracy)

KeyError: 'predicted_action'

In [ ]:
# Saving the final csv milestone 2

eval_df.to_csv(
 "../data/milestone2_output_gajendra.csv",
 index=False
)
